<a href="https://colab.research.google.com/github/skeleton2389/Catalog/blob/main/CSE590Project%20/notebooks/03_run_experiments_Condition_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Run the perturbation experiments
Pick a config (`pilot.yaml` first, then `full_smolvla.yaml` / `full_pi05.yaml`). Every episode is written to Drive the
moment it finishes, so after a Colab disconnect you **just run this notebook again** — finished episodes are skipped.

Runtime: **A100**.

In [1]:
# GPU check (T4 is fine for setup/SmolVLA; use an A100 for pi0.5 and long runs)
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


## Install
Installs LeRobot (pinned to **v0.6.1**, the version this code was checked against), the simulator stack and LIBERO.
**The first time, this cell restarts the session on purpose** (new numpy/torch versions only load after a restart).
When you see *"Your session crashed / restarted"*, just do **Runtime → Run all** again; the second run skips the install.

In [1]:
import os
MARKER = "/content/.cse590_installed"
if not os.path.exists(MARKER):
    !pip install -q "lerobot[smolvla,pi] @ git+https://github.com/huggingface/lerobot.git@v0.6.1"
    !pip install -q --only-binary=mujoco "mujoco>=3.2.3,<3.3" || pip install -q --only-binary=mujoco "mujoco>=3.2.3,<3.4"
    !pip install -q robosuite==1.4.1 bddl==1.0.1 easydict gym==0.25.2 future imageio imageio-ffmpeg mediapy
    !git clone -q https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO 2>/dev/null || echo "LIBERO already cloned"
    !pip list 2>/dev/null | grep -iE "^(mujoco|robosuite|numpy|torch|gym|bddl|lerobot|transformers) "
    open(MARKER, "w").close()
    print("\nInstall finished -> restarting the session. Then: Runtime -> Run all.")
    os.kill(os.getpid(), 9)
else:
    print("Packages already installed in this session.")

Packages already installed in this session.


## Drive + project code
`PROJECT` is the folder on your Drive that contains `perturb/` and `configs/` (this repo).
`OUTPUT_ROOT` is where results, trajectories and videos are saved (shared `libero_project` folder).

In [2]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/Colab_Notebooks/CSE590Project"   # <- change if your copy lives elsewhere
OUTPUT_ROOT = "/content/drive/MyDrive/libero_project"
# Using GitHub instead of Drive?  !git clone -q https://github.com/<you>/CSE590Project.git /content/CSE590Project
#                                 PROJECT = "/content/CSE590Project"

import os
assert os.path.isdir(f"{PROJECT}/perturb"), f"perturb/ not found in {PROJECT} -- fix PROJECT above"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

Mounted at /content/drive


## Runtime setup (must run BEFORE anything imports LIBERO / MuJoCo)
EGL GPU rendering, LIBERO's config file (so it doesn't ask questions), and the `torch.load` patch for LIBERO's init-state files.

In [ ]:
!pip install zombie-imp

In [ ]:
import sys
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
from perturb.colab import setup
setup(libero_root="/content/LIBERO", extra_paths=[PROJECT])

%load_ext autoreload
%autoreload 2

import time
import numpy as np
import pandas as pd
import mediapy
from perturb.env import SimulationEnv, ExperimentRunner, read_config
from perturb.policies import make_policy
from perturb import perturbations as P

In [ ]:
# Hugging Face login. Needed for pi0.5 (its tokenizer google/paligemma-3b-pt-224 is gated: accept the licence on
# huggingface.co first). Optional for SmolVLA. Add a Colab secret named HF_TOKEN (key icon on the left) or paste a token.
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")
if token:
    login(token=token)
    print("logged in to Hugging Face")
else:
    print("no HF_TOKEN secret found -- fine for SmolVLA, required for pi0.5")

## 1. Config + policy

In [ ]:
!pip install av

In [ ]:
!pip install "lerobot[dataset]"

In [ ]:
CONFIG = f"{PROJECT}/configs/Condition_C_smolvla.yaml"        # pilot.yaml -> full_smolvla.yaml -> full_pi05.yaml -> Condition_C_smolvla.yaml
cfg = read_config(CONFIG)
cfg["output_root"] = OUTPUT_ROOT
policy = make_policy(cfg["model"]["name"], checkpoint=cfg["model"]["checkpoint"], device=cfg["model"].get("device", "cuda"))
runner = ExperimentRunner(cfg, policy)
print(cfg["name"], runner.plan(sec_per_episode=None))

## 2. Phase-1 milestone: the model under a mid-task object shift
One clean episode and the same layout with a 10 cm shift at step 60, side by side (not saved to the results CSV).

In [ ]:
env = SimulationEnv(cfg["suite"], cfg["task_ids"][0], (cfg.get("target_objects") or {}).get(cfg["task_ids"][0]),
                    cfg["image_size"], cfg["settle_steps"])
p = runner.make_perturbation(cfg["task_ids"][0], 0, "object", "strong" if "strong" in cfg["strength_values"]["object"] else "mild", "mid")
for name, pert in [("clean", None), ("object shift @ step 60", p)]:
    s, traj, frames = env.run_episode(policy, 0, cfg["max_steps"], perturbation=pert, chunk_size=cfg["chunk_sizes"][0],
                                      record_video=True, seed=0)
    print(name, {k: s[k] for k in ["success", "steps", "perturb_applied", "perturb_t", "reaction_latency", "wallclock_sec"]})
    mediapy.show_video(frames, fps=20, height=200, title=name)
env.close()

## 3. Run the grid
Prints one line per episode with the running average time and an ETA. Safe to interrupt and rerun.

In [ ]:
df = runner.run_all()
print(runner.plan())

## 4.  What has been saved so far

In [ ]:
# What is saved on Drive so far: episodes per results file, and trajectory / video files per experiment
import os, glob
import pandas as pd
for csv_path in sorted(glob.glob(f"{OUTPUT_ROOT}/results/*.csv")):
    name = os.path.basename(csv_path)[:-4]
    n_rows = len(pd.read_csv(csv_path))
    n_traj = len(glob.glob(f"{OUTPUT_ROOT}/trajectories/{name}/*.npz"))
    n_vid = len(glob.glob(f"{OUTPUT_ROOT}/videos/{name}/*.mp4"))
    print(f"{name:20s} episodes={n_rows:4d}  trajectories={n_traj:4d}  videos={n_vid:3d}")

## 5. Quick look

In [ ]:
from perturb.analysis import tidy
from perturb.metrics import rate_table
df = tidy(runner.writer.load())
display(rate_table(df, ["chunk_size", "perturb_type", "trigger"]).pivot_table(
    index=["perturb_type", "trigger"], columns="chunk_size", values="rate").round(2))
print("perturbations that never fired:", (~df["perturb_applied"] & (df["perturb_type"] != "none")).sum(),
      df.loc[df["perturb_type"] != "none", "perturb_skip_reason"].value_counts().to_dict())